In [45]:
import pandas as pd
import numpy as np
import plotly.express as px


In [13]:
import os
os.chdir(r'C:\Users\apaks\Desktop\Real Estate Project')

In [14]:
df = pd.read_csv('artifacts\data\preprocessed-data\gurgaon_properties_post_feature_selection.csv')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\apaks\AppData\Local\Temp\ipykernel_11300\895524800.py:1: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_csv('artifacts\data\preprocessed-data\gurgaon_properties_post_feature_selection.csv')


In [15]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 83,1.18,3,3,2,relatively new,1612.0,0,0,unfurnished,budget,high-rise
1,flat,sector 62,4.00,3,3,3,new,2353.0,1,0,semifurnished,budget,high-rise
2,flat,sector 85,1.10,2,2,3+,relatively new,1459.0,0,1,unfurnished,luxury,low-rise
3,flat,sector 91,0.89,2,2,3+,relatively new,1297.0,0,0,semifurnished,budget,medium-rise
4,flat,sector 7,0.48,3,2,1,new,889.0,0,0,unfurnished,budget,low-rise


In [16]:
sorted(df['sector'].unique())

['sector 1',
 'sector 10',
 'sector 102',
 'sector 103',
 'sector 104',
 'sector 105',
 'sector 106',
 'sector 107',
 'sector 108',
 'sector 109',
 'sector 11',
 'sector 110',
 'sector 111',
 'sector 112',
 'sector 113',
 'sector 12',
 'sector 13',
 'sector 14',
 'sector 15',
 'sector 17',
 'sector 2',
 'sector 21',
 'sector 22',
 'sector 23',
 'sector 24',
 'sector 25',
 'sector 26',
 'sector 27',
 'sector 28',
 'sector 3',
 'sector 30',
 'sector 31',
 'sector 33',
 'sector 36',
 'sector 37',
 'sector 38',
 'sector 39',
 'sector 3a',
 'sector 4',
 'sector 40',
 'sector 41',
 'sector 43',
 'sector 45',
 'sector 46',
 'sector 47',
 'sector 48',
 'sector 49',
 'sector 5',
 'sector 50',
 'sector 51',
 'sector 52',
 'sector 53',
 'sector 54',
 'sector 55',
 'sector 56',
 'sector 57',
 'sector 58',
 'sector 59',
 'sector 6',
 'sector 60',
 'sector 61',
 'sector 62',
 'sector 63',
 'sector 65',
 'sector 66',
 'sector 67',
 'sector 68',
 'sector 69',
 'sector 7',
 'sector 70',
 'sector 71',
 

In [17]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv('GOOGLE_MAPS_API_KEY')

In [18]:
import requests
import json

def get_lat_long(sector, API_KEY):
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {'address': f'{sector} gurgaon',
              'key': API_KEY}
    r = requests.get(url, params)

    result_lat = r.json()['results'][0]['geometry']['location']['lat']
    result_lng = r.json()['results'][0]['geometry']['location']['lng']
    
    return result_lat, result_lng


In [19]:
get_lat_long('sector 21', API_KEY)

(28.5133929, 77.0722759)

In [20]:
all_sectors  = df['sector'].unique()

In [21]:
cordinates_dict = {}
for sector in all_sectors:
    cordinates_dict[sector] = get_lat_long(sector, API_KEY)

print(cordinates_dict)

{'sector 83': (28.3985667, 76.9646908), 'sector 62': (28.4139095, 77.08859439999999), 'sector 85': (28.40418, 76.9513432), 'sector 91': (28.4013829, 76.92253199999999), 'sector 7': (28.4643736, 77.0142942), 'sector 102': (28.4749763, 76.971507), 'sector 66': (28.3924859, 77.0540907), 'sector 25': (28.4844116, 77.0859978), 'sector 81': (28.3866865, 76.9484624), 'sector 47': (28.4275535, 77.0491679), 'sector 92': (28.407854, 76.9153281), 'sector 69': (28.3965979, 77.034108), 'sector 79': (28.3623546, 76.97870759999999), 'sector 74': (28.4157699, 77.01182460000001), 'sector 103': (28.4948731, 76.9844677), 'sector 63': (28.3971448, 77.08666459999999), 'sector 95': (28.4171926, 76.9081238), 'sector 109': (28.5072801, 77.0089452), 'sector 39': (28.4433834, 77.0515376), 'sector 111': (28.5238449, 77.0319784), 'sector 48': (28.4176955, 77.0359426), 'sector 13': (28.4756706, 77.03919669999999), 'sector 104': (28.478782, 76.9959872), 'sector 57': (28.4231993, 77.07521059999999), 'sector 2': (28.

In [22]:
cordinates_df = pd.DataFrame(cordinates_dict).T

In [23]:
cordinates_df = cordinates_df.reset_index().rename(columns = {'index': 'sector', 0: 'lat', 1:'lon'})

In [24]:
# merge original df with the cordinates df
df = df.merge(cordinates_df, on='sector')

In [25]:
df.to_csv(r'C:\Users\apaks\Desktop\Real Estate Project\artifacts\data\preprocessed-data\gurgaon_properties_with_lat_long.csv', index = False)

In [26]:
sector_wise_stats = df.groupby('sector').agg(
    avg_price_sqft = ('price', 'mean'),
    avg_builtup = ('built_up_area', 'mean'),
    lat = ('lat', 'first'),
    lon = ('lon', 'first')
).reset_index()

In [27]:
df.head()


,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category,lat,lon
0,flat,sector 83,1.18,3,3,2,relatively new,1612.0,0,0,unfurnished,budget,high-rise,28.398567,76.964691
1,flat,sector 62,4.00,3,3,3,new,2353.0,1,0,semifurnished,budget,high-rise,28.413909,77.088594
2,flat,sector 85,1.10,2,2,3+,relatively new,1459.0,0,1,unfurnished,luxury,low-rise,28.404180,76.951343
3,flat,sector 91,0.89,2,2,3+,relatively new,1297.0,0,0,semifurnished,budget,medium-rise,28.401383,76.922532
4,flat,sector 7,0.48,3,2,1,new,889.0,0,0,unfurnished,budget,low-rise,28.464374,77.014294


In [34]:
df['price_per_sqft'] = round((df['price']/df['built_up_area'])*10000000, 2)

In [35]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category,lat,lon,price_per_sqft
0,flat,sector 83,1.18,3,3,2,relatively new,1612.0,0,0,unfurnished,budget,high-rise,28.398567,76.964691,7320.10
1,flat,sector 62,4.00,3,3,3,new,2353.0,1,0,semifurnished,budget,high-rise,28.413909,77.088594,16999.58
2,flat,sector 85,1.10,2,2,3+,relatively new,1459.0,0,1,unfurnished,luxury,low-rise,28.404180,76.951343,7539.41
3,flat,sector 91,0.89,2,2,3+,relatively new,1297.0,0,0,semifurnished,budget,medium-rise,28.401383,76.922532,6861.99
4,flat,sector 7,0.48,3,2,1,new,889.0,0,0,unfurnished,budget,low-rise,28.464374,77.014294,5399.33


In [48]:
group_df = df.groupby('sector')[['price', 'price_per_sqft', 'built_up_area', 'lat', 'lon']].mean()

In [49]:
group_df.head()

,price,price_per_sqft,built_up_area,lat,lon
sector,,,,,
sector 1,1.113333,5252.311818,2101.484848,28.517613,77.045392
sector 10,1.718750,9774.307500,1887.250000,28.453693,77.000932
sector 102,1.681416,11041.299469,1546.548673,28.474976,76.971507
sector 103,1.442000,7903.454222,1804.377778,28.494873,76.984468
sector 104,1.617703,8986.439459,1841.198919,28.478782,76.995987


In [72]:
import plotly.io as pio
pio.renderers.default = "vscode"


In [73]:
fig = px.scatter_map(data_frame = group_df, lat = 'lat', lon='lon', color='price_per_sqft', size = 'built_up_area', color_continuous_scale = px.colors.cyclical.IceFire, zoom = 10, map_style='open-street-map', text = group_df.index)

fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed